In [1]:
import pandas as pd
import numpy as np
import warnings
import math
import copy
import pickle
from scipy.spatial.distance import cdist
from sklearn.preprocessing import MinMaxScaler,LabelEncoder
warnings.filterwarnings('ignore')

In [15]:
class ARING:
    
    def __init__(self,data,kappa,sigma,partial):
        self.data = data
        self.kappa = kappa
        self.sigma = sigma
        self.partial = partial
        self.attr_names = list(self.data)[:-1]
        self.attr_num = len(self.attr_names)
        self.attr_list = [x for x in range(self.attr_num)]
        self.sample_num = len(self.data)
        self.class_name = list(self.data)[-1]
        self.D_mat = None
        self.ING_list = [[] for x in range(self.attr_num)]
        self.fuzzy_list = np.zeros((self.attr_num,self.sample_num,self.sample_num))
        self.class_std_list = [[] for x in range(self.attr_num)]
        self.GI_list = [[] for x in range(self.attr_num)]
        self.attr_sort = None
        self.reduction = []
    
    def cal_OGI(self,mat_1,mat_2):
        U_mat = abs(mat_1-mat_2)
        OGI = np.sum(1-np.sum(U_mat,axis=1)/self.sample_num)/self.sample_num
        return OGI
    
    def cal_IGI(self,mat):
        IGI = 0
        for j,d_value in enumerate(self.class_value):
            mask_array = self.class_mask_p[d_value]
            mat_1 = mat[mask_array.astype(bool)]
            mat_2 = np.broadcast_to(mask_array,(self.class_v_num[j],self.sample_num))
            IGI_c = np.mean(np.sum(np.minimum(mat_1,mat_2),axis=1)/np.sum(mat_1,axis=1))
            IGI += (IGI_c/self.class_num)
        return IGI
           
    def cal_dependency(self,B_c):
        B_c_mat = np.minimum.reduce(self.fuzzy_list[B_c])
        appro_list = []
        for j,d_value in enumerate(self.class_value):
            mask_array = self.class_mask_n[d_value].astype(bool)
            d_index = np.broadcast_to(mask_array,(self.sample_num,self.sample_num))
            partial_mat = B_c_mat[d_index].reshape(self.sample_num,self.sample_num-self.class_v_num[j])
            max_r = 1-np.partition(partial_mat,-self.partial,axis=1)[:,-self.partial:]
            appro = np.sum(max_r,axis=1)/self.partial
            appro_list.append(appro)
        appro_list = np.array(appro_list).T
        B_c_dependency = sum(np.max(appro_list,axis=1))/self.sample_num
        return B_c_dependency
    
    def data_deal(self,k):
        self.data[self.class_name] = LabelEncoder().fit_transform(self.data[self.class_name])
        self.class_value = np.array(self.data[self.class_name].value_counts().index)
        self.class_v_num = np.array(self.data[self.class_name].value_counts())
        self.class_num = len(list(self.data[self.class_name].value_counts().index))
        self.judge_attr = [1 for x in range(self.attr_num)]
        self.class_mask_p = np.zeros((self.class_num,self.sample_num))
        self.class_mask_n = np.zeros((self.class_num,self.sample_num))
        for d in self.class_value:
            self.class_mask_p[d] = (self.data[self.class_name]==d).values
            self.class_mask_n[d] = (self.data[self.class_name]!=d).values
        #Use pandas to quickly calculate the integrated neighborhood distance under the single attribute granularity 
        for i,name in enumerate(self.attr_names):
            if (np.issubdtype(self.data[name],np.number)==True):
                self.data[[name]] = MinMaxScaler().fit_transform(self.data[[name]])
                self.judge_attr[i] = 1
                grouped = self.data.groupby(self.class_name)[name]
                ING_centers = grouped.transform(
                    lambda X:np.mean(
                        X.values[np.argpartition(cdist(X.values.reshape(-1,1),X.values.reshape(-1,1)),min(k-1,len(X)-1),axis=1)[:,:min(k,len(X))]],axis=1
                    )).values.reshape(-1, 1)
                self.ING_list[i] = abs(ING_centers-ING_centers.T)
                ING_std = grouped.transform(
                    lambda Y: np.std(
                        Y.values[np.argpartition(cdist(Y.values.reshape(-1,1),Y.values.reshape(-1,1)),min(k-1,len(Y)-1),axis=1)[:,:min(k,len(Y))]],axis=1
                    )).values.reshape(-1, 1)
                self.class_std_list[i] = np.exp(-abs(ING_std-ING_std.T))
            else:
                self.judge_attr[i] = 0
                prob_attr = (self.data.groupby(self.class_name)[name].value_counts(normalize=True).unstack(fill_value=0))
                sample_prob = prob_attr[self.data[name]].values.T
                self.fuzzy_list[i] = 1-(np.sum(np.abs(sample_prob[:,np.newaxis,:]-sample_prob[np.newaxis,:,:]),axis=2)/self.class_num)
                OGI = self.cal_OGI(mat_1=self.fuzzy_list[i],mat_2=self.D_mat)
                IGI = self.cal_IGI(mat=self.fuzzy_list[i])
                self.GI_list[i] = OGI+IGI
    
    def change_sigma(self):
        for i,judge in enumerate(self.judge_attr):
            if(judge!=0):
                self.judge_attr[i] = np.std(self.data[self.attr_names[i]])/self.sigma
                ING_mat = self.ING_list[i]
                ING_mat = np.where(ING_mat>self.judge_attr[i],1,ING_mat)
                self.fuzzy_list[i] = self.class_std_list[i]*(1-ING_mat)
                OGI = self.cal_OGI(mat_1=self.fuzzy_list[i],mat_2=self.D_mat)
                IGI = self.cal_IGI(mat=self.fuzzy_list[i])
                self.GI_list[i] = OGI+IGI
    
    def attribute_reduction(self):
        self.reduction.append(self.attr_sort[0])
        self.attr_sort.remove(self.attr_sort[0])
        B_dependency = self.cal_dependency(B_c=self.reduction)
        for attr in self.attr_sort:
            B_c = copy.deepcopy(self.reduction)
            B_c.append(attr)
            B_c_dependency = self.cal_dependency(B_c=B_c)
            Sig_c = B_c_dependency-B_dependency
            if(Sig_c>self.mu):
                self.reduction.append(attr)
                B_dependency = self.cal_dependency(B_c=self.reduction)
    
    def run(self):
        d_vector = self.data.values[:,-1].reshape(1,self.sample_num)
        self.D_mat = (d_vector==d_vector.T).astype(int)
        self.data_deal(k=self.kappa)
        self.change_sigma()
        
        self.omega = None
        self.mu = None 
        if(self.attr_num>1000):
            self.omega = 100
            self.mu = 0.002
        else:
            self.omega = copy.deepcopy(self.attr_num)
            self.mu = 0.005

        self.attr_sort = np.argsort(self.GI_list)[::-1][:self.omega].tolist()
        self.attribute_reduction()
        print('Attribute Importance Sequence:')
        print(self.reduction)

In [16]:
df = pd.read_csv('Connectionist.csv')

In [17]:
model = ARING(data=df,kappa=3,sigma=1,partial=3)
model.run()

Attribute Importance Sequence:
[11, 10, 9, 48, 8, 12, 47, 44, 20, 50, 35, 46, 36, 45, 19, 5, 43, 0, 42, 3, 7, 13, 53, 1, 26, 27]


In [18]:
len(model.reduction)

26

In [19]:
model.attr_num

60